In [1]:
import pandas as pd
import numpy as np
import joblib

In [2]:
model_df = pd.read_csv("/Users/jasleenkaur/Event-Driven-Congestion/data/processed/model_dataset.csv")

In [3]:
model_df.head()

,id,event_type,latitude,longitude,endlatitude,endlongitude,address,end_address,event_cause,requires_road_closure,...,month,is_weekend,priority_score,cause_score,closure_score,time_score,weekend_score,eci,congestion_risk_level,high_congestion_risk
0,FKID000000,unplanned,13.040004,77.518099,0.000000,0.000000,"Mumbai Bengaluru Highway, Jalahalli Cross Junc...",NaN,vehicle_breakdown,False,...,3.0,0,3.0,2.0,0,2,0,7.0,Medium,0
1,FKID000001,unplanned,12.921876,77.645158,0.000000,0.000000,"19th Main Road, Heavie Halcyon, Agara, HSR Lay...",NaN,vehicle_breakdown,False,...,1.0,0,3.0,2.0,0,0,0,5.0,Low,0
2,FKID000002,unplanned,12.955622,77.585708,0.000000,0.000000,"Lalbagh Main Road, Dr Sri Shantaveera Swami Ci...",NaN,others,False,...,11.0,1,1.0,1.0,0,0,1,3.0,Low,0
3,FKID000003,unplanned,13.006147,77.579435,13.006239,77.579516,"Sankey Road, Bashyam Circle, Sadashiva Nagar, ...","Sankey Road, Palace Orchard Upper, Sadashiva N...",tree_fall,True,...,3.0,0,1.0,3.0,3,2,0,9.0,Medium,0
4,FKID000004,unplanned,12.953980,77.585233,0.000000,0.000000,"Lalbagh Fort Road, Lalbagh Main Gate Junction,...",NaN,vehicle_breakdown,False,...,1.0,0,1.0,2.0,0,0,0,3.0,Low,0


In [4]:
model = joblib.load("/Users/jasleenkaur/Event-Driven-Congestion/models/high_congestion_risk_model.pkl")

In [5]:
def recommend_resources(
    prediction
):

    if prediction == 1:

        return {
            "officers":12,
            "barricades":8,
            "diversion":"Required",
            "monitoring":"Yes"
        }

    return {
        "officers":2,
        "barricades":0,
        "diversion":"Not Required",
        "monitoring":"No"
    }

In [6]:
def recommend_resources(
    prediction,
    event_cause,
    priority
):

    if prediction == 0:

        if priority == "High":
            officers = 4
        else:
            officers = 2

        barricades = 0
        diversion = "Not Required"

    else:

        officers = 10
        barricades = 6
        diversion = "Optional"

        if event_cause == "vip_movement":
            officers = 20
            barricades = 10
            diversion = "Required"

        elif event_cause == "procession":
            officers = 15
            barricades = 8
            diversion = "Required"

        elif event_cause == "protest":
            officers = 18
            barricades = 10
            diversion = "Required"

        elif event_cause == "public_event":
            officers = 12
            barricades = 8
            diversion = "Required"

        elif event_cause == "construction":
            officers = 8
            barricades = 12
            diversion = "Required"

        elif event_cause == "water_logging":
            officers = 6
            barricades = 10
            diversion = "Required"

        elif event_cause == "accident":
            officers = 8
            barricades = 4

        elif event_cause == "tree_fall":
            officers = 6
            barricades = 6

        elif event_cause == "vehicle_breakdown":
            officers = 4
            barricades = 2

        if priority == "High":
            officers += 2

    return {
        "officers": officers,
        "barricades": barricades,
        "diversion": diversion,
        "monitoring": "Yes" if prediction == 1 else "No"
    }

In [7]:
features = [
    "event_type",
    "event_cause",
    "priority",
    "requires_road_closure",
    "hour",
    "month",
    "is_weekend",
    "corridor",
    "zone"
]

In [8]:
model_df["requires_road_closure"] = model_df["requires_road_closure"].astype(int)

In [9]:
X = model_df[features]
model_df["prediction"] = model.predict(X)

In [10]:
recommendations = model_df.apply(
    lambda row:
    recommend_resources(
        row["prediction"],
        row["event_cause"],
        row["priority"]
    ),
    axis=1
)

In [11]:
recommendation_df = pd.json_normalize(
    recommendations
)

recommendation_df.head()

,officers,barricades,diversion,monitoring
0,4,0,Not Required,No
1,4,0,Not Required,No
2,2,0,Not Required,No
3,6,6,Optional,Yes
4,2,0,Not Required,No


In [12]:
final_df = pd.concat(
    [
        model_df,
        recommendation_df
    ],
    axis=1
)

In [13]:
final_df[
    [
        "event_cause",
        "priority",
        "prediction",
        "officers",
        "barricades",
        "diversion"
    ]
].head(20)

,event_cause,priority,prediction,officers,barricades,diversion
0,vehicle_breakdown,High,0,4,0,Not Required
1,vehicle_breakdown,High,0,4,0,Not Required
2,others,Low,0,2,0,Not Required
3,tree_fall,Low,1,6,6,Optional
4,vehicle_breakdown,Low,0,2,0,Not Required
5,accident,Low,0,2,0,Not Required
6,vehicle_breakdown,Low,0,2,0,Not Required
7,others,Low,0,2,0,Not Required
8,vehicle_breakdown,Low,0,2,0,Not Required
9,water_logging,High,0,4,0,Not Required


In [14]:
final_df.to_csv(
    "../data/processed/recommendation_dataset.csv",
    index=False
)